# mCREAM Ensemble Analysis

Analysis notebook for the ensemble experiments (`mcream_ensemble_main.py`).

**Experiment matrix:**
- `average` — hard expert + simple average (baseline)
- `weighted` — hard expert + learnable π (graph-level attention)
- `soft_average` — soft-edge α + simple average (edge-level learning)
- `soft_weighted` — soft-edge α + learnable π (full proposed method)

**Noise families:** deletion / addition / reversal × low / medium / high

**Datasets:** Complete_Concept_FMNIST, CelebA

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path
import torch
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

# ── CONFIGURE ────────────────────────────────────────────────────────────────
EXPERIMENTS_ROOT  = Path("/home/dani00003/mCREAM/experiments")
GRAPHS_ROOT       = Path("/home/dani00003/mCREAM/data")
DAG_CFMNIST       = Path("/home/dani00003/mCREAM/data/FashionMNIST/Complete_Concept_FMNIST_DAG.csv")
DAG_CELEBA        = Path("/home/dani00003/mCREAM/data/CelebA/final_DAG_unfair.csv")

DATASETS    = ["Complete_Concept_FMNIST", "CelebA"]
ACTIONS     = ["deletion", "addition", "reversal"]
LEVELS      = ["low", "medium", "high"]
ETYPES      = ["average", "weighted", "soft_average", "soft_weighted"]
ETYPE_LABEL = {
    "average":      "Avg (hard)",
    "weighted":     "Weighted π (hard)",
    "soft_average": "Avg (soft α)",
    "soft_weighted":"Weighted α+π (full)",
}
ACTION_COLOR = {"deletion": "#e74c3c", "addition": "#2ecc71", "reversal": "#3498db"}
ETYPE_MARKER = {"average": "o", "weighted": "s", "soft_average": "^", "soft_weighted": "D"}

print(f"Experiments root exists: {EXPERIMENTS_ROOT.exists()}")

## 1. Ground Truth DAGs

In [ ]:
def plot_dag(dag_path: Path, title: str, num_concepts: int):
    if not dag_path.exists():
        print(f"DAG not found: {dag_path}"); return
    df = pd.read_csv(dag_path, index_col=0)
    vals = (df.values != 0).astype(float)
    # Show only concept rows/cols for clarity
    u2c = vals[:num_concepts, :num_concepts]
    c2y = vals[num_concepts:, :num_concepts]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    concepts = df.columns[:num_concepts].tolist()
    tasks    = df.columns[num_concepts:].tolist()

    sns.heatmap(u2c, ax=axes[0], cmap="Blues", vmin=0, vmax=1,
                xticklabels=concepts, yticklabels=concepts,
                annot=True, fmt=".0f", linewidths=0.5, annot_kws={"size": 8})
    axes[0].set_title("Concept→Concept (u2c)")

    sns.heatmap(c2y, ax=axes[1], cmap="Blues", vmin=0, vmax=1,
                xticklabels=concepts, yticklabels=tasks,
                annot=True, fmt=".0f", linewidths=0.5, annot_kws={"size": 7})
    axes[1].set_title("Concept→Task (c2y)")

    fig.suptitle(f"Ground Truth DAG — {title}", fontsize=13, fontweight="bold")
    plt.tight_layout()
    plt.show()
    print(f"  u2c edges: {int(u2c.sum())}  |  c2y edges: {int(c2y.sum())}")

plot_dag(DAG_CFMNIST, "Complete Concept FMNIST", num_concepts=11)
plot_dag(DAG_CELEBA,  "CelebA", num_concepts=7)

## 2. Expert Graph Examples (one per action × noise level)

In [ ]:
def plot_expert_graphs(dataset_key: str, num_concepts: int, num_classes: int,
                       dag_path: Path, graphs_base: Path):
    """
    For each action × level, load expert_0's u2c and c2y graphs and
    show them alongside the ground truth.
    """
    if not dag_path.exists(): return
    gt_df = pd.read_csv(dag_path, index_col=0)
    concepts = gt_df.columns[:num_concepts].tolist()
    tasks    = gt_df.columns[num_concepts:].tolist()

    gt_u2c = (gt_df.values[:num_concepts, :num_concepts] != 0).astype(float)
    gt_c2y = (gt_df.values[num_concepts:, :num_concepts] != 0).astype(float)

    for action in ACTIONS:
        fig, axes = plt.subplots(len(LEVELS) + 1, 2,
                                 figsize=(12, 3.5 * (len(LEVELS) + 1)))
        fig.suptitle(f"{dataset_key} — {action}-only noise, expert_0",
                     fontsize=12, fontweight="bold")

        # Row 0: ground truth
        for ax, mat, yl, title in [
            (axes[0,0], gt_u2c, concepts, "GT u2c"),
            (axes[0,1], gt_c2y, tasks,    "GT c2y"),
        ]:
            sns.heatmap(mat, ax=ax, cmap="Blues", vmin=0, vmax=1,
                        xticklabels=concepts, yticklabels=yl,
                        linewidths=0.3, annot_kws={"size":7})
            ax.set_title(title, fontweight="bold")

        for row, level in enumerate(LEVELS, start=1):
            graph_dir = graphs_base / dataset_key / "expert_graphs" / "ensemble" / f"{action}_{level}"
            u2c_file = graph_dir / "u2c" / "expert_0.pt"
            c2y_file = graph_dir / "c2y" / "expert_0.pt"

            if not u2c_file.exists():
                axes[row, 0].set_title(f"{level} — not found")
                axes[row, 1].set_visible(False)
                continue

            u2c = torch.load(u2c_file, weights_only=True).float().numpy()
            c2y = torch.load(c2y_file, weights_only=True).float().numpy()[:, :num_concepts]

            for ax, mat, yl, gt_mat, side in [
                (axes[row,0], u2c, concepts, gt_u2c, "u2c"),
                (axes[row,1], c2y, tasks,    gt_c2y, "c2y"),
            ]:
                changed = (mat.astype(bool) != gt_mat.astype(bool)).sum()
                total   = gt_mat.size
                sns.heatmap(mat, ax=ax, cmap="Blues", vmin=0, vmax=1,
                            xticklabels=concepts, yticklabels=yl,
                            linewidths=0.3, annot_kws={"size":7})
                ax.set_title(f"{level} {side}  ({changed}/{total} changed = {100*changed/total:.1f}%)")

        plt.tight_layout()
        plt.show()

plot_expert_graphs("FashionMNIST", 11, 10, DAG_CFMNIST, GRAPHS_ROOT)
plot_expert_graphs("CelebA",       7,  1,  DAG_CELEBA,  GRAPHS_ROOT)

## 3. Load All Ensemble Results

In [ ]:
def load_ensemble_results(root: Path) -> pd.DataFrame:
    """
    Scan mCREAM_Ensemble experiment directories and load all per-seed CSVs.
    Parses experiment_name like: ensemble_average_deletion_medium
    """
    rows = []
    for dataset in DATASETS:
        ens_dir = root / dataset / "train_cbm" / "mCREAM_Ensemble"
        if not ens_dir.exists():
            print(f"  [SKIP] {ens_dir}")
            continue
        for exp_dir in sorted(ens_dir.iterdir()):
            if not exp_dir.is_dir(): continue
            exp_name = exp_dir.name  # ensemble_average_deletion_medium
            parts = exp_name.split("_")
            # parse: ensemble_{soft_}{etype}_{action}_{level}
            try:
                # find etype: first part after 'ensemble'
                idx = 1  # skip 'ensemble'
                soft = ""
                if parts[idx] == "soft":
                    soft = "soft_"
                    idx += 1
                etype   = soft + parts[idx];  idx += 1
                action  = parts[idx];          idx += 1
                level   = "_".join(parts[idx:])
            except IndexError:
                etype = action = level = "unknown"

            for seed_dir in sorted(exp_dir.glob("seed_*/lightning_logs/version_*")):
                seed = int(seed_dir.parent.parent.name.split("_")[1])
                for csv_f in sorted(seed_dir.glob("*.csv")):
                    if any(x in csv_f.name for x in
                           ["perc_", "_set_", "intervention_results",
                            "per_expert", "exogenous", "correlation"]):
                        continue
                    try:
                        df = pd.read_csv(csv_f)
                        df["dataset"]     = dataset
                        df["exp_name"]    = exp_name
                        df["etype"]       = etype
                        df["action"]      = action
                        df["noise_level"] = level
                        df["seed"]        = seed
                        rows.append(df)
                    except Exception as e:
                        print(f"  ERR {csv_f}: {e}")

    if not rows:
        print("No ensemble results found. Jobs may still be running.")
        return pd.DataFrame()

    df = pd.concat(rows, ignore_index=True)
    print(f"Loaded {len(df)} rows")
    print(f"  datasets:    {sorted(df['dataset'].unique())}")
    print(f"  etype:       {sorted(df['etype'].unique())}")
    print(f"  actions:     {sorted(df['action'].unique())}")
    print(f"  noise_level: {sorted(df['noise_level'].unique())}")
    return df

ens_df = load_ensemble_results(EXPERIMENTS_ROOT)
ens_df.head(3) if len(ens_df) > 0 else print("Empty — check paths or wait for jobs.")

## 4. Summary Table — Task Accuracy, CCI, PFI, C2Y Baseline

In [ ]:
if len(ens_df) == 0:
    print("No data yet.")
else:
    KEY_METRICS = [
        "test_task_accuracy", "test_concept_accuracy",
        "CCI", "PFI_concept_importance", "PFI_side_importance",
        "c2y_baseline_accuracy", "concept_leakage",
    ]
    available = [c for c in KEY_METRICS if c in ens_df.columns]
    GROUP = ["dataset", "action", "noise_level", "etype"]

    means  = ens_df.groupby(GROUP)[available].mean()
    stds   = ens_df.groupby(GROUP)[available].std()
    counts = ens_df.groupby(GROUP)[available[0]].count().rename("n_seeds")

    # Format as mean ± std
    summary = pd.DataFrame(index=means.index)
    summary["n_seeds"] = counts
    for col in available:
        summary[col] = (means[col].map("{:.4f}".format)
                        + " ± " + stds[col].map("{:.4f}".format))

    print("SUMMARY TABLE — mean ± std across seeds")
    print("=" * 80)
    # Show one dataset at a time for readability
    for ds in DATASETS:
        subset = summary.xs(ds, level="dataset") if ds in summary.index.get_level_values("dataset") else None
        if subset is not None and len(subset) > 0:
            print(f"\n{'─'*60}\n  {ds}\n{'─'*60}")
            display(subset)

## 5. Per-Expert Accuracy vs Ensemble Accuracy

In [ ]:
def load_per_expert_preds(root: Path) -> pd.DataFrame:
    """Load per_expert_predictions.csv files and extract accuracy per expert."""
    rows = []
    for dataset in DATASETS:
        ens_dir = root / dataset / "train_cbm" / "mCREAM_Ensemble"
        if not ens_dir.exists(): continue
        for csv_f in ens_dir.rglob("expert_predictions/per_expert_predictions.csv"):
            try:
                df = pd.read_csv(csv_f)
                # Extract metadata from path
                parts = csv_f.parts
                exp_name = None
                seed = None
                for i, p in enumerate(parts):
                    if p == "mCREAM_Ensemble" and i+1 < len(parts):
                        exp_name = parts[i+1]
                    if p.startswith("seed_"):
                        seed = int(p.split("_")[1])

                if exp_name is None: continue
                p = exp_name.split("_")
                idx = 1
                soft = ""
                if p[idx] == "soft": soft = "soft_"; idx += 1
                etype  = soft + p[idx]; idx += 1
                action = p[idx];        idx += 1
                level  = "_".join(p[idx:])

                # Compute accuracy per expert
                expert_cols = [c for c in df.columns if c.endswith("_correct") and c != "ensemble_correct"]
                for col in expert_cols:
                    m_idx = col.split("_")[1]  # y_0_correct → 0
                    rows.append({
                        "dataset": dataset, "exp_name": exp_name,
                        "etype": etype, "action": action, "noise_level": level,
                        "seed": seed, "expert": int(m_idx),
                        "expert_accuracy": df[col].mean(),
                        "ensemble_accuracy": df["ensemble_correct"].mean(),
                    })
            except Exception as e:
                print(f"ERR {csv_f}: {e}")

    return pd.DataFrame(rows)

expert_acc_df = load_per_expert_preds(EXPERIMENTS_ROOT)
print(f"Loaded {len(expert_acc_df)} expert-level rows")

In [ ]:
if len(expert_acc_df) == 0:
    print("No per-expert data yet.")
else:
    # For each dataset × action × noise_level: show individual expert accuracy
    # vs ensemble accuracy — shows how ensemble recovers from poor individual experts
    for dataset in DATASETS:
        ds_df = expert_acc_df[expert_acc_df["dataset"] == dataset]
        if ds_df.empty: continue

        for action in ACTIONS:
            act_df = ds_df[ds_df["action"] == action]
            if act_df.empty: continue

            fig, axes = plt.subplots(1, len(LEVELS), figsize=(5*len(LEVELS), 5), sharey=True)
            fig.suptitle(f"{dataset} — {action} noise\nPer-expert vs Ensemble accuracy",
                         fontsize=12, fontweight="bold")

            for ax, level in zip(axes, LEVELS):
                lv_df = act_df[act_df["noise_level"] == level]
                if lv_df.empty:
                    ax.set_title(f"{level} — no data"); continue

                # Mean across seeds
                grp = lv_df.groupby(["etype", "expert"]).agg(
                    acc_mean=("expert_accuracy", "mean"),
                    acc_std=("expert_accuracy",  "std"),
                    ens_mean=("ensemble_accuracy","mean")
                ).reset_index()

                for etype, sub in grp.groupby("etype"):
                    color  = plt.cm.Set2(ETYPES.index(etype) / len(ETYPES))
                    marker = ETYPE_MARKER.get(etype, "o")
                    ax.errorbar(sub["expert"], sub["acc_mean"], yerr=sub["acc_std"],
                                label=ETYPE_LABEL.get(etype, etype),
                                marker=marker, color=color, linestyle="--",
                                capsize=3, markersize=7)
                    # Ensemble line (horizontal)
                    ens_acc = sub["ens_mean"].mean()
                    ax.axhline(ens_acc, color=color, linestyle="-", alpha=0.6,
                               linewidth=2)

                ax.set_title(f"{level}")
                ax.set_xlabel("Expert index")
                ax.set_ylabel("Accuracy")
                ax.legend(fontsize=7)

            plt.tight_layout()
            plt.show()

## 6. Intervention Curves

In [ ]:
def load_ensemble_interventions(root: Path) -> pd.DataFrame:
    rows = []
    for dataset in DATASETS:
        ens_dir = root / dataset / "train_cbm" / "mCREAM_Ensemble"
        if not ens_dir.exists(): continue
        for csv_f in ens_dir.rglob("intervention_results.csv"):
            try:
                df = pd.read_csv(csv_f)
                parts = csv_f.parts
                exp_name = seed = None
                for i, p in enumerate(parts):
                    if p == "mCREAM_Ensemble" and i+1 < len(parts): exp_name = parts[i+1]
                    if p.startswith("seed_"): seed = int(p.split("_")[1])
                if exp_name is None: continue
                p = exp_name.split("_")
                idx = 1
                soft = ""
                if p[idx] == "soft": soft = "soft_"; idx += 1
                etype  = soft + p[idx]; idx += 1
                action = p[idx];        idx += 1
                level  = "_".join(p[idx:])
                df["dataset"] = dataset; df["etype"] = etype
                df["action"]  = action;  df["noise_level"] = level
                df["seed"]    = seed
                rows.append(df)
            except Exception as e:
                print(f"ERR {csv_f}: {e}")
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()

interv_df = load_ensemble_interventions(EXPERIMENTS_ROOT)
print(f"Loaded {len(interv_df)} intervention rows")

In [ ]:
if len(interv_df) == 0:
    print("No intervention data yet.")
else:
    # Intervention curves: one figure per dataset × action
    # Rows = noise levels, cols = ensemble types
    # Style matches the CREAM paper (Figure 6)
    for dataset in DATASETS:
        for action in ACTIONS:
            sub = interv_df[(interv_df["dataset"] == dataset) &
                            (interv_df["action"]  == action)]
            if sub.empty: continue

            n_levels = len(LEVELS)
            n_etypes = len(ETYPES)
            fig, axes = plt.subplots(n_levels, 1,
                                     figsize=(9, 4 * n_levels), sharex=False)
            if n_levels == 1: axes = [axes]

            fig.suptitle(f"{dataset} — {action} noise\nIntervention Curves",
                         fontsize=13, fontweight="bold")

            for ax, level in zip(axes, LEVELS):
                lv = sub[sub["noise_level"] == level]
                if lv.empty:
                    ax.set_title(f"{level} — no data"); continue

                agg = (lv.groupby(["etype", "num_interventions"])["test_task_accuracy"]
                         .agg(["mean", "std"])
                         .reset_index())

                for i_et, etype in enumerate(ETYPES):
                    et_data = agg[agg["etype"] == etype]
                    if et_data.empty: continue
                    color  = plt.cm.tab10(i_et)
                    marker = ETYPE_MARKER.get(etype, "o")
                    ax.plot(et_data["num_interventions"], et_data["mean"],
                            label=ETYPE_LABEL.get(etype, etype),
                            color=color, marker=marker, markersize=5)
                    ax.fill_between(et_data["num_interventions"],
                                   et_data["mean"] - et_data["std"],
                                   et_data["mean"] + et_data["std"],
                                   alpha=0.15, color=color)

                ax.set_title(f"{level} disagreement")
                ax.set_xlabel("Number of interventions")
                ax.set_ylabel("Task Accuracy")
                ax.legend(fontsize=8, loc="lower right")
                ax.axhline(1.0, color="black", linestyle=":", alpha=0.4)

            plt.tight_layout()
            plt.savefig(f"ensemble_interventions_{dataset}_{action}.png",
                        dpi=150, bbox_inches="tight")
            plt.show()

## 7. Task Accuracy: etype × noise level × action

In [ ]:
if len(ens_df) == 0:
    print("No data yet.")
else:
    for dataset in DATASETS:
        ds = ens_df[ens_df["dataset"] == dataset]
        if ds.empty: continue

        fig, axes = plt.subplots(1, len(ACTIONS), figsize=(6*len(ACTIONS), 5), sharey=True)
        fig.suptitle(f"Task Accuracy — {dataset}", fontsize=13, fontweight="bold")

        level_order = ["low", "medium", "high"]
        palette = {et: plt.cm.tab10(i) for i, et in enumerate(ETYPES)}

        for ax, action in zip(axes, ACTIONS):
            act = ds[ds["action"] == action]
            if act.empty: ax.set_title(f"{action} — no data"); continue

            sns.boxplot(
                data=act, x="noise_level", y="test_task_accuracy",
                hue="etype", order=level_order,
                hue_order=ETYPES, palette=palette, ax=ax
            )
            ax.set_title(f"{action}")
            ax.set_xlabel("Noise level")
            ax.set_ylabel("Task accuracy")
            ax.legend(title="Method", fontsize=7, title_fontsize=7)

        plt.tight_layout()
        plt.savefig(f"ensemble_task_acc_{dataset}.png", dpi=150, bbox_inches="tight")
        plt.show()

## 8. Learned Expert Weights π (weighted & soft_weighted only)

In [ ]:
if len(ens_df) == 0:
    print("No data yet.")
elif "expert_weights" not in ens_df.columns:
    print("expert_weights column not found.")
else:
    import ast
    w_df = ens_df[ens_df["etype"].isin(["weighted", "soft_weighted"])].copy()
    w_df = w_df.dropna(subset=["expert_weights"])

    if w_df.empty:
        print("No expert weight data yet.")
    else:
        # Parse string list → actual list
        def parse_weights(x):
            try: return ast.literal_eval(x) if isinstance(x, str) else x
            except: return None
        w_df["weights"] = w_df["expert_weights"].apply(parse_weights)
        w_df = w_df.dropna(subset=["weights"])

        rows_w = []
        for _, row in w_df.iterrows():
            for m_idx, w in enumerate(row["weights"]):
                rows_w.append({"dataset": row["dataset"], "etype": row["etype"],
                                "action": row["action"], "noise_level": row["noise_level"],
                                "seed": row["seed"], "expert": m_idx, "weight": w})
        weights_long = pd.DataFrame(rows_w)

        for dataset in DATASETS:
            ds = weights_long[weights_long["dataset"] == dataset]
            if ds.empty: continue

            fig, axes = plt.subplots(2, len(ACTIONS),
                                     figsize=(5*len(ACTIONS), 8))
            fig.suptitle(f"Learned Expert Weights π — {dataset}",
                         fontsize=12, fontweight="bold")

            for col, action in enumerate(ACTIONS):
                for row_i, etype in enumerate(["weighted", "soft_weighted"]):
                    ax = axes[row_i, col]
                    sub = ds[(ds["action"] == action) & (ds["etype"] == etype)]
                    if sub.empty:
                        ax.set_title(f"{etype}/{action} — no data"); continue

                    agg = sub.groupby(["noise_level", "expert"])["weight"].mean().reset_index()
                    pivot = agg.pivot(index="expert", columns="noise_level", values="weight")
                    pivot = pivot.reindex(columns=LEVELS)
                    pivot.plot(kind="bar", ax=ax, colormap="viridis", legend=(col==0))
                    ax.set_title(f"{ETYPE_LABEL.get(etype, etype)}\n{action}")
                    ax.set_xlabel("Expert")
                    ax.set_ylabel("Mean π weight")
                    ax.axhline(1/len(pivot), color="red", linestyle="--",
                               alpha=0.5, label="uniform")

            plt.tight_layout()
            plt.show()

## 9. CCI & PFI Summary

In [ ]:
if len(ens_df) == 0:
    print("No data yet.")
else:
    cci_cols = [c for c in ["CCI", "PFI_concept_importance", "PFI_side_importance"]
                if c in ens_df.columns]
    if not cci_cols:
        print("CCI/PFI columns not found yet.")
    else:
        for dataset in DATASETS:
            ds = ens_df[ens_df["dataset"] == dataset].dropna(subset=cci_cols[:1])
            if ds.empty: continue

            fig, axes = plt.subplots(1, len(cci_cols), figsize=(6*len(cci_cols), 5))
            if len(cci_cols) == 1: axes = [axes]
            fig.suptitle(f"Concept Importance — {dataset}", fontsize=12, fontweight="bold")

            for ax, metric in zip(axes, cci_cols):
                sub = ds.dropna(subset=[metric])
                if sub.empty: continue
                sns.boxplot(data=sub, x="etype", y=metric,
                            hue="action", order=ETYPES, ax=ax,
                            palette=ACTION_COLOR)
                ax.set_title(metric)
                ax.set_xticklabels([ETYPE_LABEL.get(e, e) for e in ETYPES],
                                   rotation=25, ha="right")
                ax.set_xlabel("")
                if metric == "CCI":
                    ax.axhline(0.5, color="red", linestyle="--", alpha=0.5,
                               label="0.5 threshold")

            plt.tight_layout()
            plt.show()

## 10. Robustness Plot: Accuracy vs Noise Level (H1)

In [ ]:
if len(ens_df) == 0:
    print("No data yet.")
else:
    # Key plot for H1: as noise increases, which method degrades least?
    level_num = {"low": 0.25, "medium": 0.50, "high": 0.75}

    for dataset in DATASETS:
        ds = ens_df[ens_df["dataset"] == dataset].copy()
        if ds.empty: continue
        ds["noise_prob"] = ds["noise_level"].map(level_num)
        ds = ds.dropna(subset=["noise_prob", "test_task_accuracy"])

        fig, axes = plt.subplots(1, len(ACTIONS), figsize=(5*len(ACTIONS), 5), sharey=True)
        fig.suptitle(f"H1 Robustness — {dataset}\nTask Accuracy vs Noise Level",
                     fontsize=12, fontweight="bold")

        for ax, action in zip(axes, ACTIONS):
            act = ds[ds["action"] == action]
            if act.empty: continue

            agg = act.groupby(["etype", "noise_prob"])["test_task_accuracy"].agg(
                ["mean", "std"]).reset_index()

            for i_et, etype in enumerate(ETYPES):
                sub = agg[agg["etype"] == etype]
                if sub.empty: continue
                color  = plt.cm.tab10(i_et)
                marker = ETYPE_MARKER.get(etype, "o")
                ax.plot(sub["noise_prob"], sub["mean"],
                        label=ETYPE_LABEL.get(etype, etype),
                        color=color, marker=marker, markersize=7, linewidth=2)
                ax.fill_between(sub["noise_prob"],
                                sub["mean"] - sub["std"],
                                sub["mean"] + sub["std"],
                                alpha=0.15, color=color)

            ax.set_title(f"{action}")
            ax.set_xlabel("Noise probability")
            ax.set_ylabel("Task Accuracy")
            ax.set_xticks([0.25, 0.50, 0.75])
            ax.set_xticklabels(["low\n(0.25)", "medium\n(0.50)", "high\n(0.75)"])
            ax.legend(fontsize=7)

        plt.tight_layout()
        plt.savefig(f"ensemble_robustness_{dataset}.png", dpi=150, bbox_inches="tight")
        plt.show()

## 11. Export LaTeX-ready Summary Table

In [ ]:
if len(ens_df) > 0:
    key = ["test_task_accuracy", "test_concept_accuracy", "CCI",
           "PFI_concept_importance", "intervention_acc_max"]
    available = [c for c in key if c in ens_df.columns]

    agg = (ens_df.groupby(["dataset", "action", "noise_level", "etype"])[available]
                 .agg(["mean", "std"])
                 .round(4))
    agg.to_csv("mcream_ensemble_summary.csv")
    print("Saved: mcream_ensemble_summary.csv")
    display(agg)